# Sudoku board frame debug split

Lekki notebook diagnostyczny oparty o lokalne moduly w `src/MachineLearning/draft`. Tutaj zostaje tylko konfiguracja i wizualizacja.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

DRAFT_DIR = Path("/home/wojtek/projects/sudoku/src/MachineLearning/draft")
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from sudoku_board_debug_binary_variants import (
    BinaryCleanupSettings,
    build_binary_variants,
)
from sudoku_board_debug_core import (
    BoardDebugSettings,
    binarize_image,
    load_image,
    preprocess_image,
)
from sudoku_board_debug_line_experiment import (
    LineMergeSettings,
    describe_merged_candidates,
    resolve_line_merge_settings,
    run_line_detection_experiment,
)
from sudoku_board_debug_notebook import (
    apply_exif_orientation,
    read_exif_orientation_label,
    show_display_image,
)
from sudoku_board_debug_visualization import (
    draw_family_overlay,
    draw_merged_candidates_overlay,
    draw_raw_segments_overlay,
    show_image,
)

plt.rcParams["figure.figsize"] = (18, 10)
plt.rcParams["image.cmap"] = "gray"


In [ ]:
# Wymagane biblioteki:
# /twoj/python -m pip install numpy matplotlib opencv-python


In [ ]:
IMAGE_PATH = Path("/home/wojtek/projects/sudoku/data/raw/boards/v2_test/image1041.jpg")

board_settings = BoardDebugSettings(
    grayscale_color_conversion_code=cv2.COLOR_BGR2GRAY,
    gaussian_kernel_size=(5, 5),
    gaussian_sigma_x=0.0,
    adaptive_threshold_block_size=11,
    adaptive_threshold_c=2,
)
line_settings = LineMergeSettings()
cleanup_settings = BinaryCleanupSettings()

print(f"IMAGE_PATH = {IMAGE_PATH}")
print("Notebook skupia sie tylko na znajdowaniu linii po adaptive threshold.")


In [ ]:
source_image = load_image(IMAGE_PATH)
preprocessed_image = preprocess_image(source_image, board_settings)
binary_image = binarize_image(preprocessed_image, board_settings)
# Algorytm pracuje na THRESH_BINARY_INV, ale do podgladu pokazujemy
# binarke w naturalniejszej polaryzacji: ciemna siatka na jasnym tle.
binary_display_image = cv2.bitwise_not(binary_image)
binary_debug_image = cv2.cvtColor(binary_display_image, cv2.COLOR_GRAY2BGR)

image_height, image_width = binary_image.shape
minimum_dimension = min(image_height, image_width)
resolved_line_settings = resolve_line_merge_settings(binary_image.shape, line_settings)

print("Binary-only debug starts from adaptive threshold.")
print(f"Image size: {image_width}x{image_height}")
print(f"Raw Hough threshold: {resolved_line_settings.raw_hough_threshold}")
print(f"Raw min line length px: {resolved_line_settings.raw_min_line_length_px}")
print(f"Raw max line gap px: {resolved_line_settings.raw_max_line_gap_px}")
print(
    "Post-merge projection distance px:",
    resolved_line_settings.post_merge_projection_distance_px,
)
print(
    "Post-merge endpoint gap px:",
    resolved_line_settings.post_merge_endpoint_gap_px,
)
print(
    "Post-merge min overlap ratio:",
    resolved_line_settings.post_merge_min_overlap_ratio,
)
print(f"Min merged span px: {resolved_line_settings.min_merged_span_px}")
print("Source image is now EXIF-normalized inside load_image().")

figure, axes = plt.subplots(1, 3, figsize=(18, 6))
show_image(axes[0], source_image, "Source image", is_bgr=True)
show_image(axes[1], preprocessed_image, "Grayscale + blur")
show_image(axes[2], binary_display_image, "Adaptive threshold")
figure.tight_layout()


In [ ]:
# Glowna logika eksperymentu zostala wyniesiona do modulow .py,
# zeby notebook zostal lekki i szybki do iteracji:
# - sudoku_board_debug_line_experiment.py
# - sudoku_board_debug_binary_variants.py
# - sudoku_board_debug_notebook.py


## Cel eksperymentu

Na tym etapie notebook szuka tylko wszystkich linii siatki. Ramke planszy wybierzemy dopiero pozniej, kiedy lista wykrytych i sklejonych linii bedzie stabilna.

In [ ]:
base_result = run_line_detection_experiment(
    "adaptive only",
    binary_image,
    line_settings,
)

raw_segments_overlay = draw_raw_segments_overlay(
    binary_debug_image,
    base_result.raw_segments,
)
family_overlay_image = draw_family_overlay(
    binary_debug_image,
    base_result.primary_segments,
    base_result.secondary_segments,
)
segment_merge_overlay_image = draw_merged_candidates_overlay(
    binary_debug_image,
    base_result.primary_filtered_candidates,
    base_result.secondary_filtered_candidates,
)
final_merge_overlay_image = draw_merged_candidates_overlay(
    binary_debug_image,
    base_result.primary_final_candidates,
    base_result.secondary_final_candidates,
)
final_merge_overlay_on_source_image = draw_merged_candidates_overlay(
    source_image,
    base_result.primary_final_candidates,
    base_result.secondary_final_candidates,
)

primary_angle_label = (
    f"{base_result.primary_angle_degrees:.2f}"
    if base_result.primary_angle_degrees is not None
    else "n/a"
)
secondary_angle_label = (
    f"{base_result.secondary_angle_degrees:.2f}"
    if base_result.secondary_angle_degrees is not None
    else "n/a"
)

print("Line finding experiment")
print("Szukanie wszystkich linii po adaptive threshold, bez wybierania ramki.")
print(f"Raw segments: {len(base_result.raw_segments)}")
print(
    "Primary family:",
    f"{len(base_result.primary_segments)} segments,",
    f"angle={primary_angle_label}",
)
print(
    "Secondary family:",
    f"{len(base_result.secondary_segments)} segments,",
    f"angle={secondary_angle_label}",
)
print("Merged primary candidates:", len(base_result.primary_merged_candidates))
print("Merged secondary candidates:", len(base_result.secondary_merged_candidates))
print("Filtered primary candidates:", len(base_result.primary_filtered_candidates))
print("Filtered secondary candidates:", len(base_result.secondary_filtered_candidates))
print(
    "Final primary candidates after post-merge:",
    len(base_result.primary_final_candidates),
)
print(
    "Final secondary candidates after post-merge:",
    len(base_result.secondary_final_candidates),
)

print("\nPrimary candidates after first merge:")
for description in describe_merged_candidates(base_result.primary_filtered_candidates):
    print("  ", description)

print("\nSecondary candidates after first merge:")
for description in describe_merged_candidates(base_result.secondary_filtered_candidates):
    print("  ", description)

print("\nPrimary candidates after post-merge:")
for description in describe_merged_candidates(base_result.primary_final_candidates):
    print("  ", description)

print("\nSecondary candidates after post-merge:")
for description in describe_merged_candidates(base_result.secondary_final_candidates):
    print("  ", description)

if (
    len(base_result.primary_final_candidates) != 10
    or len(base_result.secondary_final_candidates) != 10
):
    print("\nUwaga: po post-merge nadal nie mamy idealnego ukladu 10x10.")
    print(
        "To jest oczekiwane na tym etapie: notebook ma pomoc pokazac,"
        " ktore kandydaty dalej sa zdublowane albo za slabe."
    )

figure, axes = plt.subplots(2, 4, figsize=(28, 12))
show_display_image(
    axes[0, 0],
    source_image,
    "Source image",
    display_exif_orientation_label,
    is_bgr=True,
)
show_display_image(
    axes[0, 1],
    preprocessed_image,
    "Grayscale + blur",
    display_exif_orientation_label,
)
show_display_image(
    axes[0, 2],
    binary_display_image,
    "Adaptive threshold (display)",
    display_exif_orientation_label,
)
show_display_image(
    axes[0, 3],
    raw_segments_overlay,
    "Raw Hough segments",
    display_exif_orientation_label,
    is_bgr=True,
)
show_display_image(
    axes[1, 0],
    family_overlay_image,
    "Two dominant angle families",
    display_exif_orientation_label,
    is_bgr=True,
)
show_display_image(
    axes[1, 1],
    segment_merge_overlay_image,
    "After segment merge",
    display_exif_orientation_label,
    is_bgr=True,
)
show_display_image(
    axes[1, 2],
    final_merge_overlay_image,
    "After candidate post-merge",
    display_exif_orientation_label,
    is_bgr=True,
)
show_display_image(
    axes[1, 3],
    final_merge_overlay_on_source_image,
    "Final candidates on source",
    display_exif_orientation_label,
    is_bgr=True,
)
figure.tight_layout()


## Eksperyment: czyszczenie binarki przed szukaniem linii

Ten blok porownuje kilka wariantow przygotowania obrazu po `adaptive threshold`, zanim uruchomimy Hough i dalsze scalanie kandydatow linii.

Cel jest prosty: zachowac dlugie struktury poziome i pionowe siatki, ale ograniczyc drobny foregroundowy szum, ktory produkuje zbyt wiele krotkich segmentow.

In [ ]:
resolved_cleanup_settings, line_variants = build_binary_variants(
    binary_image,
    minimum_dimension,
    cleanup_settings,
)
variant_results = [
    run_line_detection_experiment(name, variant_binary, line_settings)
    for name, variant_binary in line_variants
]

print("Binary cleanup experiment before line search")
print(
    "Connected components min area px:",
    resolved_cleanup_settings.min_component_area_px,
)
print(
    "Directional open kernel length:",
    resolved_cleanup_settings.open_kernel_length,
)
print(
    "Directional close kernel length:",
    resolved_cleanup_settings.close_kernel_length,
)

for result in variant_results:
    primary_angle_label = (
        f"{result.primary_angle_degrees:.2f}"
        if result.primary_angle_degrees is not None
        else "n/a"
    )
    secondary_angle_label = (
        f"{result.secondary_angle_degrees:.2f}"
        if result.secondary_angle_degrees is not None
        else "n/a"
    )
    primary_count = len(result.primary_final_candidates)
    secondary_count = len(result.secondary_final_candidates)

    print(f"\nVariant: {result.name}")
    print("  Raw segments:", len(result.raw_segments))
    print(
        "  Primary family:",
        len(result.primary_segments),
        f"angle={primary_angle_label}",
    )
    print(
        "  Secondary family:",
        len(result.secondary_segments),
        f"angle={secondary_angle_label}",
    )
    print("  Final primary candidates:", primary_count)
    print("  Final secondary candidates:", secondary_count)
    print(
        "  Target >= 10 x 10:",
        "YES" if primary_count >= 10 and secondary_count >= 10 else "NO",
    )

figure, axes = plt.subplots(len(variant_results), 4, figsize=(24, 6 * len(variant_results)))
if len(variant_results) == 1:
    axes = np.array([axes])

for row_index, result in enumerate(variant_results):
    binary_variant_display = cv2.bitwise_not(result.binary)
    binary_variant_bgr = cv2.cvtColor(binary_variant_display, cv2.COLOR_GRAY2BGR)
    raw_segments_overlay = draw_raw_segments_overlay(
        binary_variant_bgr,
        result.raw_segments,
    )
    family_overlay = draw_family_overlay(
        binary_variant_bgr,
        result.primary_segments,
        result.secondary_segments,
    )
    final_overlay_on_source = draw_merged_candidates_overlay(
        source_image,
        result.primary_final_candidates,
        result.secondary_final_candidates,
    )

    show_display_image(
        axes[row_index, 0],
        binary_variant_display,
        f"{result.name}\nbinary (display)",
        display_exif_orientation_label,
    )
    show_display_image(
        axes[row_index, 1],
        raw_segments_overlay,
        f"{result.name}\nraw Hough",
        display_exif_orientation_label,
        is_bgr=True,
    )
    show_display_image(
        axes[row_index, 2],
        family_overlay,
        f"{result.name}\nangle families",
        display_exif_orientation_label,
        is_bgr=True,
    )
    show_display_image(
        axes[row_index, 3],
        final_overlay_on_source,
        (
            f"{result.name}\n"
            f"final P={len(result.primary_final_candidates)} / "
            f"S={len(result.secondary_final_candidates)}"
        ),
        display_exif_orientation_label,
        is_bgr=True,
    )

figure.tight_layout()
